In [25]:
!pip install nilearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.8 MB/s eta 0:00:00


In [27]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, roc_curve)
from nilearn import datasets, plotting

In [30]:
dir = './outputs'
seed = 42
n_folds = 5
n_trees = 500
top_k = 20
n_rois = 200

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [26]:
X_train = np.load(f'/content/X_train.npy')
X_test  = np.load(f'/content/X_test.npy')
y_train = np.load(f'/content/y_train.npy')
y_test  = np.load(f'/content/y_test.npy')

In [31]:
cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
rf = RandomForestClassifier(n_estimators=n_trees, max_features='sqrt',
                             class_weight='balanced', n_jobs=-1, random_state=seed)

fold_acc, fold_auc, fold_f1 = [], [], []
all_y_true, all_y_prob = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), 1):
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_train[tr_idx])
    Xval = sc.transform(X_train[val_idx])
    rf.fit(Xtr, y_train[tr_idx])
    prob = rf.predict_proba(Xval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    fold_acc.append(accuracy_score(y_train[val_idx], pred))
    fold_auc.append(roc_auc_score(y_train[val_idx], prob))
    fold_f1.append(f1_score(y_train[val_idx], pred))
    all_y_true.extend(y_train[val_idx])
    all_y_prob.extend(prob)
    print(f"Fold {fold} | Acc {fold_acc[-1]:.3f}  AUC {fold_auc[-1]:.3f}  F1 {fold_f1[-1]:.3f}")

all_y_true = np.array(all_y_true)
all_y_prob = np.array(all_y_prob)
tn, fp, fn, tp = confusion_matrix(all_y_true, (all_y_prob>=0.5).astype(int)).ravel()
print(f"\nCV  Acc {np.mean(fold_acc):.3f}±{np.std(fold_acc):.3f}  AUC {np.mean(fold_auc):.3f}±{np.std(fold_auc):.3f}  F1 {np.mean(fold_f1):.3f}±{np.std(fold_f1):.3f}")
print(f"CV  Sensitivity = {tp/(tp+fn):.3f}   Specificity = {tn/(tn+fp):.3f}")

Fold 1 | Acc 0.662  AUC 0.733  F1 0.489
Fold 2 | Acc 0.570  AUC 0.624  F1 0.419
Fold 3 | Acc 0.603  AUC 0.681  F1 0.491
Fold 4 | Acc 0.617  AUC 0.752  F1 0.471
Fold 5 | Acc 0.674  AUC 0.754  F1 0.582

CV  Acc 0.625±0.038  AUC 0.708±0.050  F1 0.490±0.053
CV  Sensitivity = 0.393   Specificity = 0.824


In [32]:
sc_final   = StandardScaler()
X_train_sc = sc_final.fit_transform(X_train)
X_test_sc  = sc_final.transform(X_test)
rf.fit(X_train_sc, y_train)
test_prob  = rf.predict_proba(X_test_sc)[:, 1]
test_pred  = (test_prob >= 0.5).astype(int)
print(f"\nTest  Acc {accuracy_score(y_test, test_pred):.3f}  AUC {roc_auc_score(y_test, test_prob):.3f}  F1 {f1_score(y_test, test_pred):.3f}")


Test  Acc 0.638  AUC 0.727  F1 0.529


In [33]:
np.save(f'{OUTPUT_DIR}/rf_feature_importances.npy', rf.feature_importances_)
pd.DataFrame({'fold': range(1, n_folds +1), 'accuracy': fold_acc,
              'auc': fold_auc, 'f1': fold_f1}).to_csv(
    f'{OUTPUT_DIR}/rf_cv_results.csv', index=False)

In [35]:
roi_i, roi_j = np.triu_indices(n_rois, k=1)

background  = shap.sample(X_train_sc, 200, random_state=seed)
explainer   = shap.TreeExplainer(rf, background)
shap_values_output = explainer.shap_values(X_test_sc)

# Ensure shap_asd is a 2D array (N_samples, N_features) for the positive class
# shap.TreeExplainer.shap_values typically returns a list of arrays for binary classification
# but the kernel state shows shap_asd as a 3D array directly.
if isinstance(shap_values_output, list):
    shap_asd = shap_values_output[1] # Get SHAP values for the positive class (class 1)
else:
    # If it's not a list, it must be a 3D array like the kernel state indicates.
    # Select the positive class (index 1) from the last dimension.
    shap_asd = shap_values_output[:, :, 1]

mean_shap   = np.abs(shap_asd).mean(axis=0)
top_idx     = np.argsort(mean_shap)[::-1][:top_k]
top_shap    = mean_shap[top_idx]
top_roi_i   = roi_i[top_idx]
top_roi_j   = roi_j[top_idx]

np.save(f'{OUTPUT_DIR}/shap_values_asd.npy', shap_asd)
pd.DataFrame({'rank': range(1, top_k +1), 'feature_idx': top_idx,
              'mean_shap': top_shap, 'roi_i': top_roi_i,
              'roi_j': top_roi_j}).to_csv(
    f'{OUTPUT_DIR}/top_shap_features.csv', index=False)

print(f"\nTop {top_k} connections saved \u2192 top_shap_features.csv")
for k in range(top_k):
    print(f"  Rank {k+1:>2}  ROI{top_roi_i[k]+1:>3} - ROI{top_roi_j[k]+1:<3}  {top_shap[k]:.6f}")

100%|===================| 353/354 [00:43<00:00]       


Top 20 connections saved → top_shap_features.csv
  Rank  1  ROI 18 - ROI107  0.001793
  Rank  2  ROI 19 - ROI140  0.001627
  Rank  3  ROI 45 - ROI127  0.001299
  Rank  4  ROI 17 - ROI104  0.001246
  Rank  5  ROI140 - ROI174  0.001207
  Rank  6  ROI 18 - ROI69   0.001116
  Rank  7  ROI117 - ROI148  0.001116
  Rank  8  ROI104 - ROI140  0.001016
  Rank  9  ROI 18 - ROI66   0.001007
  Rank 10  ROI 45 - ROI65   0.000987
  Rank 11  ROI 47 - ROI94   0.000966
  Rank 12  ROI  5 - ROI140  0.000939
  Rank 13  ROI140 - ROI191  0.000934
  Rank 14  ROI 16 - ROI139  0.000915
  Rank 15  ROI 18 - ROI150  0.000890
  Rank 16  ROI 53 - ROI56   0.000886
  Rank 17  ROI 52 - ROI107  0.000878
  Rank 18  ROI 22 - ROI145  0.000862
  Rank 19  ROI 48 - ROI117  0.000844
  Rank 20  ROI 18 - ROI117  0.000803


In [37]:
power  = datasets.fetch_coords_power_2011()
coords = np.column_stack([power.rois['x'], power.rois['y'], power.rois['z']])[:n_rois]

conn   = np.zeros((n_rois, n_rois))
for k in range(top_k):
    i, j = top_roi_i[k], top_roi_j[k]
    conn[i, j] = conn[j, i] = top_shap[k]

display = plotting.plot_connectome(conn, coords, node_size=25,
                                   edge_threshold='80%', edge_cmap='YlOrRd',
                                   node_color='steelblue', display_mode='ortho',
                                   colorbar=True,
                                   title=f'Top {top_k} SHAP Connections')
display.savefig(f'{OUTPUT_DIR}/glass_brain_ortho.png')
display.close()